# 04_evaluation: Final Exam on 2016 Loans
Compare both final models on the untouched test set, read the results in business terms, and beat the simple grade rule (REQ-04).

In [0]:
%run ./00_config

## Step 1: Load the test set and both final models from MLflow

In [0]:
import mlflow, mlflow.spark
from pyspark.sql import functions as F

test = spark.table(TEST_TABLE)
mlflow.set_experiment(EXPERIMENT_PATH)

def load_final(run_name):
    """Find the latest run with this name in our experiment, and load its saved model."""
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT_PATH],
                              filter_string=f"tags.mlflow.runName = '{run_name}'",
                              order_by=["start_time DESC"])
    run_id = runs.iloc[0]["run_id"]
    model = mlflow.spark.load_model(f"runs:/{run_id}/model",
                                    dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/artifacts/mlflow_tmp")
    return run_id, model

gbt_run_id, gbt_model = load_final("final_gbt")
lr_run_id,  lr_model  = load_final("final_logreg")
print(f"Test loans: {test.count():,}\nGBT run: {gbt_run_id}\nLR run:  {lr_run_id}")

Loaded test set (98,301 loans from 2016) and both final models (GBT and Logistic Regression) from MLflow.

## Step 2: Final scores on the 2016 test set (both models)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

def get_metrics(preds):
    """AUC-ROC, AUC-PR, precision, recall, F1 for the default class (copied from 03)."""
    auc_roc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(preds)
    auc_pr  = BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(preds)
    c = preds.agg(
        F.sum(((F.col("prediction") == 1) & (F.col("label") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("prediction") == 1) & (F.col("label") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("prediction") == 0) & (F.col("label") == 1)).cast("int")).alias("fn"),
    ).first()
    precision = c.tp / (c.tp + c.fp) if (c.tp + c.fp) else 0.0
    recall    = c.tp / (c.tp + c.fn) if (c.tp + c.fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall, "f1": f1}

# Predict 2016, keeping each loan's probability of default
gbt_preds = gbt_model.transform(test).withColumn("p_default", vector_to_array("probability")[1])
lr_preds  = lr_model.transform(test).withColumn("p_default", vector_to_array("probability")[1])

results = [("GBT", get_metrics(gbt_preds)), ("Logistic Regression", get_metrics(lr_preds))]
display(spark.createDataFrame([(name, *[round(v, 4) for v in m.values()]) for name, m in results],
                              ["model", "auc_roc", "auc_pr", "precision", "recall", "f1"]))

**Test set (2016):** GBT AUC-ROC **0.706**, recall **68.3%**, precision **33.3%**, F1 0.448. Logistic Regression is slightly behind (0.703).
- **AUC-ROC:** in ~71% of defaulter vs good-borrower pairs, the model ranks the defaulter as riskier.
- **Recall:** the model catches ~7 of every 10 real defaults.
- **Precision:** about 1 in 3 flagged loans really defaults; the other 2 would have repaid.
AUC dropped only slightly from validation (0.712 → 0.706), so the model generalizes well to a new year.
The higher recall / lower precision comes from training on 2015 (26% default rate), which makes the model more cautious.

## Step 3: Confusion matrix (GBT) with loan amounts per box

In [0]:
# Count loans and total loan amount in each of the 4 boxes
cm = (gbt_preds
      .groupBy("label", "prediction")
      .agg(F.count("*").alias("loans"),
           F.round(F.sum("loan_amnt") / 1e6, 1).alias("loan_amount_millions"))
      .orderBy("label", "prediction"))
display(cm)

**Confusion matrix (GBT, 2016, threshold 0.5):**
| | Approve | Reject |
|---|---|---|
| Repaid | 47,235 ($623.9M) ✅ | 29,481 ($480.4M) ❌ good borrowers lost |
| Defaulted | 6,854 ($95.3M) ❌ losses | 14,731 ($246.8M) ✅ losses avoided |

- **False negative** (approved defaulter): the lender loses most of the money lent, the most expensive mistake.
- **False positive** (rejected good borrower): the lender loses the interest profit and the customer.
At threshold 0.5 the model rejects ~45% of applicants, too strict for most lenders. The threshold should be chosen
based on business costs (next step).

## Step 4: Precision-Recall trade-off across cut-offs (GBT)

In [0]:
# For each cut-off, compute precision, recall and the % of applicants rejected
n_test = gbt_preds.count()
rows = []
for t in [round(0.10 + 0.05 * i, 2) for i in range(17)]:          # 0.10, 0.15, ... 0.90
    c = gbt_preds.agg(
        F.sum(((F.col("p_default") >= t) & (F.col("label") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("p_default") >= t) & (F.col("label") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("p_default") <  t) & (F.col("label") == 1)).cast("int")).alias("fn"),
    ).first()
    precision = c.tp / (c.tp + c.fp) if (c.tp + c.fp) else 0.0
    recall    = c.tp / (c.tp + c.fn)
    rows.append((t, round(precision, 4), round(recall, 4), round((c.tp + c.fp) / n_test, 4)))

pr_table = spark.createDataFrame(rows, ["cutoff", "precision", "recall", "reject_rate"])
display(pr_table)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Precision-Recall trade-off (GBT, 2016):** lowering the cut-off catches more defaulters but rejects more good borrowers.
- 0.50 (default): rejects 45%, catches 68% of defaults, precision 33%
- **0.60:** rejects 29%, catches 50% of defaults, precision 38%
- 0.70: rejects 14%, catches 27% of defaults, precision 44%

**Choosing the cut-off is a business decision:** it depends on how many applicants the lender can reject and on the
cost of a default vs a lost good customer. Example: at 0.60, 38% of rejected loans are real defaults vs 22% at random
(~1.7× better than random selection).

## Step 5: Model vs the simple grade rule, at the same reject rate (fair comparison)

In [0]:
def pr_at(df, reject_expr):
    """Precision, recall and reject rate when loans matching reject_expr are rejected."""
    c = df.agg(
        F.sum((reject_expr & (F.col("label") == 1)).cast("int")).alias("tp"),
        F.sum((reject_expr & (F.col("label") == 0)).cast("int")).alias("fp"),
        F.sum((~reject_expr & (F.col("label") == 1)).cast("int")).alias("fn"),
    ).first()
    return c.tp / (c.tp + c.fp), c.tp / (c.tp + c.fn), (c.tp + c.fp) / n_test

rules = {"reject G": ["G"], "reject F-G": ["F", "G"],
         "reject E-G": ["E", "F", "G"], "reject D-G": ["D", "E", "F", "G"]}
rows = []
for name, grades in rules.items():
    r_prec, r_rec, rate = pr_at(gbt_preds, F.col("grade").isin(grades))        # the old rule
    t = gbt_preds.approxQuantile("p_default", [1 - rate], 0.001)[0]            # model cut-off giving the same reject rate
    m_prec, m_rec, _ = pr_at(gbt_preds, F.col("p_default") >= t)               # the model
    rows.append((name, round(rate, 4), round(r_prec, 4), round(r_rec, 4), round(m_prec, 4), round(m_rec, 4)))

display(spark.createDataFrame(rows, ["rule", "reject_rate", "rule_precision", "rule_recall",
                                     "model_precision", "model_recall"]))

**Model vs grade rule (same reject rate, 2016):** the model wins at every level.
At ~29% rejected ("reject grades D–G"): recall **50.0% vs 47.8%**, precision **38.3% vs 36.6%**, i.e. ~490 more
defaulters caught and ~490 fewer good borrowers rejected, roughly $8M of risky loan volume avoided per year,
with no extra rejections.
The gain is modest because `grade` is already a strong risk score built with data we don't have (e.g. FICO).
The model adds value on top of it using income, debt, loan term and employment details.

## Step 6: Top 10 feature importances (GBT)

In [0]:
# Importance of each of the 45 inputs, from the GBT stage (the last stage of the pipeline)
importances = gbt_model.stages[-1].featureImportances.toArray()

# Get the name of each of the 45 positions from the "features" column's metadata
attrs = gbt_preds.schema["features"].metadata["ml_attr"]["attrs"]
names = {a["idx"]: a["name"] for group in attrs.values() for a in group}

top10 = sorted([(names[i], round(float(v), 4)) for i, v in enumerate(importances)],
               key=lambda x: -x[1])[:10]
display(spark.createDataFrame(top10, ["feature", "importance"]))

Databricks visualization. Run in Databricks to view.

**What drives defaults (GBT feature importance):**
1. **Interest rate (21.8%)**, by far the strongest: Lending Club prices loans by its own risk assessment, so the rate carries
   hidden credit information (and makes `grade` redundant, which is why grade is not in the top 10).
2. **Affordability:** debt-to-income (7.6%), income (7.5%), loan amount (6.4%), installment (3.2%).
3. **Stability:** mortgage accounts (6.0%), loan term (4.8%), and our engineered flag **`emp_length_missing` (4.6%)**:
   an unknown job length is itself a risk signal.
Importance shows usefulness, not direction.

# Evaluation Summary
- **Test set (2016, never seen in training):** GBT AUC-ROC **0.706**, recall 68.3%, precision 33.3% (cut-off 0.5).
  Logistic Regression is close behind (0.703). Accuracy is not used because of class imbalance.
- **Confusion matrix:** at 0.5 the model rejects ~45% of applicants, too strict. Missed defaults cost the lent money;
  wrongly rejected good borrowers cost the interest profit.
- **Cut-off is a business choice:** e.g. 0.60 rejects ~29%, catches 50% of defaults, with 38% precision (vs 22% at random).
- **Beats the simple grade rule** at every reject rate. At ~29% rejected: ~490 more defaulters caught, same number of rejections.
- **Top drivers:** interest rate, debt-to-income, income, loan amount, mortgage accounts, term, unknown job length.